In [26]:
# Step 1: Setup
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [20]:
import pandas as pd
import os
from tqdm import tqdm

# Paths
data_dir = "/content/drive/MyDrive/DTSC 5082 Datasets"
discharge_path = os.path.join(data_dir, "cleaned_discharge.csv")
mimic_path = os.path.join(data_dir, "mimic-iv-bhc.csv")

# Load datasets
discharge_df = pd.read_csv(discharge_path)
mimic_df = pd.read_csv(mimic_path)
pd.set_option('display.max_colwidth', None)

In [21]:
discharge_df.head(1)

note_id  subject_id   hadm_id note_type  note_seq  \
0  10000032-DS-21    10000032  22595853        DS        21   

             charttime            storetime  \
0  2180-05-07 00:00:00  2180-05-09 15:26:00   

                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                            

In [22]:
mimic_df.head(1)

note_id  \
0  10000032-DS-21   

                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                               

In [23]:
# Take first 1000 rows
subset_df = discharge_df.iloc[:1000].copy()

In [24]:
# Step 2: Install and Load T5
!pip install -q transformers evaluate bert_score

In [25]:
import torch
from transformers import T5Tokenizer, T5ForConditionalGeneration

device = "cuda" if torch.cuda.is_available() else "cpu"
model_name = "t5-base"

tokenizer = T5Tokenizer.from_pretrained(model_name)
model = T5ForConditionalGeneration.from_pretrained(model_name).to(device)

# Step 3: Summary Generation Function
def generate_summary(text):
    text = str(text).strip()
    if len(text) < 30:
        return ""

    input_text = "summarize: " + text
    inputs = tokenizer(input_text, return_tensors="pt", max_length=512, truncation=True).to(device)

    summary_ids = model.generate(
        **inputs,
        max_length=200,
        num_beams=4,
        early_stopping=True
    )

    return tokenizer.decode(summary_ids[0], skip_special_tokens=True)


In [27]:
# Step 4: Apply to first 1000 rows
tqdm.pandas()
subset_df['generated_summary'] = subset_df['text'].progress_apply(generate_summary)
subset_df.head(3)


100%|██████████| 1000/1000 [22:16<00:00,  1.34s/it]


note_id  subject_id   hadm_id note_type  note_seq  \
0  10000032-DS-21    10000032  22595853        DS        21   
1  10000032-DS-22    10000032  22841357        DS        22   
2  10000032-DS-23    10000032  29079034        DS        23   

             charttime            storetime  \
0  2180-05-07 00:00:00  2180-05-09 15:26:00   
1  2180-06-27 00:00:00  2180-07-01 10:15:00   
2  2180-07-25 00:00:00  2180-07-25 21:42:00   

                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                

In [28]:
# Step 5: Match with MIMIC using note_id
merged_df = subset_df.merge(mimic_df[['note_id', 'target']], on='note_id', how='inner')

# Step 6: Filter only where generated_summary is not empty
eval_df = merged_df[merged_df['generated_summary'].str.strip().astype(bool)]

In [29]:
!pip install rouge_score

  Preparing metadata (setup.py) ... done
  Created wheel for rouge_score: filename=rouge_score-0.1.2-py3-none-any.whl size=24934 sha256=ec98b671eff719dd32d0e6ac24199b1f9655c08ecedafdd1666055db2b63da78
  Stored in directory: /root/.cache/pip/wheels/1e/19/43/8a442dc83660ca25e163e1bd1f89919284ab0d0c1475475148
Successfully built rouge_score


In [30]:
# Step 7: Evaluate Metrics
import evaluate
from bert_score import score as bert_score

# ROUGE
rouge = evaluate.load("rouge")
rouge_result = rouge.compute(
    predictions=eval_df['generated_summary'].tolist(),
    references=eval_df['target'].tolist()
)

# BLEU
bleu = evaluate.load("bleu")
bleu_result = bleu.compute(
    predictions=eval_df['generated_summary'].tolist(),
    references=[[ref] for ref in eval_df['target'].tolist()]
)

# BERTScore
P, R, F1 = bert_score(
    eval_df['generated_summary'].tolist(),
    eval_df['target'].tolist(),
    lang='en'
)

bertscore_result = {
    "precision": P.mean().item(),
    "recall": R.mean().item(),
    "f1": F1.mean().item()
}


tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/482 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


model.safetensors:   0%|          | 0.00/1.42G [00:00<?, ?B/s]

Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [31]:
# Step 8: Display Scores
print("Number of rows evaluated:", len(eval_df))
print("ROUGE:", rouge_result)
print("BLEU:", bleu_result)
print("BERTScore:", bertscore_result)

Number of rows evaluated: 819
ROUGE: {'rouge1': np.float64(0.09519244511238645), 'rouge2': np.float64(0.028151832873676612), 'rougeL': np.float64(0.06936576020863697), 'rougeLsum': np.float64(0.06936515131630819)}
BLEU: {'bleu': 6.87569790025609e-07, 'precisions': [0.5129861982434128, 0.12736228711245615, 0.06018120494676278, 0.04078442035142576], 'brevity_penalty': 6.11000211949018e-06, 'length_ratio': 0.07689005301265261, 'translation_length': 31880, 'reference_length': 414618}
BERTScore: {'precision': 0.8366767764091492, 'recall': 0.7773444056510925, 'f1': 0.8057137131690979}
